# L5b: Multiple Asset Geometric Brownian Motion
In the last few lectures we've modeled the share price of a single firm using binomial (or multinomial) lattices and geometric Brownian motion (GBM). The question that we explore today is how do we extend these ideas to several firms whose prices may move together? 

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * **Model correlated asset prices:** Write the multiple asset GBM model, explain how a covariance factor introduces correlated fluctuations, and use the exact one-step transition to simulate prices.
> * **Estimate and interpret covariance:** Calculate the covariance matrix from historical growth rates, interpret its entries, and convert it to the covariance rate used by the price model.
> * **Explore portfolio weights:** Relate initial investment weights to buy-and-hold wealth and explain how the Dirichlet distribution generates long-only allocations with different degrees of concentration.

Firms respond to common market news and economic conditions, so we need to describe the relationships between their price movements. In this lecture, we use a covariance matrix to represent those relationships and extend the single asset model to correlated assets.

Let's get started!

___

## Examples
Today, we will be using the following examples to illustrate key concepts:

> [▶ Compute the covariance matrix for our dataset](CHEME-5660-L5b-Example-CovarianceMatrix-Fall-2026.ipynb). How do we measure whether firms' growth rates move together? We compute the covariance matrix from historical growth rates, convert it to the covariance rate used by the multiple asset model, and verify the corresponding volatilities against our L4b estimates. We then examine a pair of firms to interpret their covariance and correlation.

> [▶ Sample portfolio weights with the Dirichlet distribution](CHEME-5660-L5b-Example-Dirichlet-PortfolioWeights-Fall-2026.ipynb). How does dividing our investment among firms affect portfolio growth and risk? We sample long-only allocations using the Dirichlet distribution, examine how its concentration parameters shape the weights, and compare the estimated growth and risk of the sampled portfolios. We also track buy-and-hold wealth to see how the fractions invested in each firm change as prices move.

Optional examples on covariance estimation and changing correlations are listed in the Optional Advanced Material section at the end of this lecture.

___


## Company Profile: Bridgewater Associates

[Bridgewater Associates](https://www.bridgewater.com/) is a global macro investment firm that manages portfolios for pension funds, endowments, and sovereign wealth funds. [Ray Dalio](https://www.bridgewater.com/our-founder) founded the firm in 1975 and led it as CEO, chief investment officer, and chairman before stepping down as chairman at the end of 2021. Its process is systematic: the firm writes down cause-and-effect rules for how economies and markets work, then tests them against historical data.

Bridgewater is an allocator, and its best-known portfolio is built on how assets move together.

> __What makes Bridgewater distinctive?__
>
> * __All Weather:__ Launched in 1996, [All Weather](https://www.bridgewater.com/research-and-insights/the-all-weather-story) is designed to hold up whether growth and inflation come in above or below expectations. Stocks, nominal bonds, inflation-linked bonds, and commodities respond differently to the same news, so the portfolio balances risk, not dollars, across them. Dalio's article [Engineering Targeted Returns and Risks](https://bridgewater.brightspotcdn.com/fa/e3/d09e72bd401a8414c5c0bdaf88bb/bridgewater-associates-engineering-targeted-returns-and-risks-aug-2011.pdf) explains the principles, now widely known as __risk parity__.
> * __Pure Alpha:__ Launched in 1991, Pure Alpha is the firm's actively traded portfolio. Its returns are meant to be uncorrelated with the asset classes themselves, so Bridgewater separates active return (alpha) from the return of holding the markets (beta) and sizes each independently.
> * __Culture:__ The firm's [culture](https://www.bridgewater.com/culture) is built on what Dalio calls radical truth and radical transparency, described in his book [*Principles: Life and Work*](https://www.principles.com/).

__Explore further:__

* __Jobs and internships:__ Browse the [job openings](https://www.bridgewater.com/working-at-bridgewater/job-openings) and [students](https://www.bridgewater.com/working-at-bridgewater/students) pages. The [2027 Investment Associate Internship Program](https://www.bridgewater.com/2027-investment-associate-internship-program), an eight-week summer program in Westport, Connecticut, is posted as of September 21, 2026.
* __YouTube:__ Watch [How The Economic Machine Works](https://www.youtube.com/watch?v=PHe0bXAIuk0), a thirty-minute animated explanation of credit and debt cycles narrated by Dalio on the [Principles by Ray Dalio](https://www.youtube.com/@principlesbyraydalio) channel, or visit the firm's own [Bridgewater Associates](https://www.youtube.com/@Bridgewater) channel.

__Connection to today's lecture:__ All Weather rests on the observation that a portfolio's risk depends on how its assets move together, not only on each asset's risk. Today we make that idea precise at classroom scale: we estimate a covariance matrix from historical growth rates, extend the price model to reproduce it, and sample long-only weights to see how the allocation changes portfolio growth and risk. Let's begin by recalling the single asset model.

___

## Multiple Asset Geometric Brownian Motion (MAGBM) Model
Simulating $M$ single asset geometric Brownian motion (GBM) models independently assumes zero relationship between the assets' growth rates. To describe firms that respond together to market news, or perhaps respond in opposite directions, we need to capture the relationships between their price movements. We do this by introducing a covariance matrix that describes how the assets' growth rates move together.

Consider a portfolio $\mathcal{P}=\{1,2,\ldots,M\}$ of $M\geq2$ assets, with share prices $S_i(t)>0$ and initial prices $S_i(0)>0$. Here, we index the assets by $i,j\in\mathcal{P}$. Like single asset models, each asset has its own drift parameter $\mu_i$ (units: inverse years). However, the different between single and multiple asset models comes in the volatility. 

> __Covariance rate matrix__ 
> 
> For multiple assets, we introduce a __covariance rate matrix__ $\mathbf{C}\in\mathbb{R}^{M\times M}$ (units: inverse years) that captures the individual volatilities of each asset, as well as the relationships between assets. The diagonal entries of the covariance rate matrix are the squared volatilities, $C_{ii}=\sigma_i^2$, while the off-diagonal entries describe the __covariance__, i.e., the relationship between two assets' growth rates per unit time. We assume the drifts and covariance rate remain constant over the modeled period.
>
> The covariance rate matrix is symmetric and positive semi-definite, ensuring that the resulting price paths are realistic.

Let $W_1(t),\ldots,W_M(t)$ be independent standard Wiener processes, indexed by $\ell\in\{1,2,\ldots,M\}$. We combine their increments using a __loading matrix__ $\mathbf{A}\in\mathbb{R}^{M\times M}$ whose entries have units of inverse square-root years. We choose the loading matrix $\mathbf{A}$ to reproduce the covariance rate through the identity:
$$
\mathbf{A}\mathbf{A}^{\top}=\mathbf{C}.
$$
The multiple asset GBM model for the share price of asset $i$ is then given by:
$$
\frac{dS_i(t)}{S_i(t)}
=\mu_i\,dt+\underbrace{\sum_{\ell=1}^{M}A_{i\ell}\,dW_\ell(t)}_{\text{correlated noise}},
\qquad i\in\mathcal{P}.
$$
Thus, each asset's price evolution is influenced by the correlated noise term, which captures the interdependencies between the assets. Each row of $\mathbf{A}$ determines how the independent Wiener increments contribute to one asset's price fluctuations, i.e., the $i$-th row of $\mathbf{A}$ gives the weights for the $M$ independent Wiener increments that drive asset $i$'s price.

Before we discuss how to choose the loading matrix $\mathbf{A}$, let's examine why the $\mathbf{A}\mathbf{A}^{\top}$ factorization is even necessary.

> __Why do we need the $\mathbf{A}\mathbf{A}^{\top}$ factorization?__
>
> * __Independent noise becomes correlated noise:__ Every asset uses the same vector of Wiener increments, combined with its own row of $\mathbf{A}$. These shared random inputs allow the assets' fluctuations to be correlated, the price fluctuations of one asset to influence the price fluctuations of another.
>
> * __The factor reproduces the covariance rate:__ Independent Wiener increments have variance $dt$ and zero covariance with one another. The covariance of the noise terms for assets $i$ and $j$ is therefore given by:
> $$
> \begin{aligned}
> \operatorname{Cov}\!\left(\sum_{\ell=1}^{M}A_{i\ell}\,dW_\ell,
> \sum_{\ell=1}^{M}A_{j\ell}\,dW_\ell\right)
> &=\sum_{\ell=1}^{M}A_{i\ell}A_{j\ell}\,dt\\
> &=(\mathbf{A}\mathbf{A}^{\top})_{ij}\,dt=C_{ij}\,dt.
> \end{aligned}
> $$
> Thus, choosing $\mathbf{A}\mathbf{A}^{\top}=\mathbf{C}$ gives the model the covariance per unit time specified by the $\mathbf{C}$ matrix.

For a positive definite $\mathbf{C}$, a __Cholesky decomposition__ gives a lower-triangular factor $\mathbf{A}$. The symmetric positive semidefinite square root $\mathbf{C}^{1/2}$ is another valid choice, including when $\mathbf{C}$ is singular. We will examine these covariance-matrix properties in the next section.

### Exact one-step transition
Applying Itô's lemma to $\ln S_i(t)$ gives each asset its own half-variance correction. The [derivation notebook](advanced/ito-derivation/CHEME-5660-L5b-Derivation-MAGBM-Solution-Fall-2026.ipynb) extends the single asset argument from L4b to several noise sources; here we record the one new step. The correction is half the quadratic variation of the noise term per unit time. Because the Wiener processes are independent, $dW_\ell\,dW_m=0$ for $\ell\neq m$ and $dW_\ell^2=dt$, so:
$$
\Bigl(\sum_{\ell=1}^{M}A_{i\ell}\,dW_\ell\Bigr)^2
=\sum_{\ell=1}^{M}\sum_{m=1}^{M}A_{i\ell}A_{im}\,dW_\ell\,dW_m
=\sum_{\ell=1}^{M}A_{i\ell}^{2}\,dt.
$$
Following L4b's notation, the __mean growth rate__ of asset $i$ is given by:
$$
\mu_{g,i}
=\mu_i-\frac{1}{2}\sum_{\ell=1}^{M}A_{i\ell}^{2}
=\mu_i-\frac{C_{ii}}{2}.
$$
The sum of squares of row $i$ of $\mathbf{A}$ is the $(i,i)$ entry of $\mathbf{A}\mathbf{A}^{\top}$, so $\sum_{\ell=1}^{M}A_{i\ell}^{2}=(\mathbf{A}\mathbf{A}^{\top})_{ii}=C_{ii}$. Because $C_{ii}=\sigma_i^2$, this is the same correction used in the single asset model, and $\mu_i=\mu_{g,i}+C_{ii}/2$.

As in L4b, define the time grid $t_k=k\Delta t$ for $k=0,1,\ldots,N$, where $\Delta t>0$ is measured in years and $T=N\Delta t$. Let $\mathbf{Z}_k\sim\mathcal{N}(\mathbf{0},\mathbf{I}_M)$ be a vector of $M$ independent standard normal random variables (shocks) drawn at step $k$. The exact one-step transition is given by:
$$
\boxed{
S_i(t_k)
=S_i(t_{k-1})\exp\!\left[\mu_{g,i}\Delta t
+\sqrt{\Delta t}\,(\mathbf{A}\mathbf{Z}_k)_i\right],
\qquad i\in\mathcal{P},\quad k=1,2,\ldots,N.
}
$$
Here, $(\mathbf{A}\mathbf{Z}_k)_i=\sum_{\ell=1}^{M}A_{i\ell}Z_{k,\ell}$ is the noise contribution for asset $i$ before scaling by $\sqrt{\Delta t}$. We use the __same vector $\mathbf{Z}_k$ for all assets within step $k$__ and draw a __new independent vector at every step__, so the shocks $\mathbf{Z}_1,\ldots,\mathbf{Z}_N$ are independent. This preserves the covariance between assets and the independent increments through time.

For constant parameters, the transition gives exact model prices at the grid points. As in the single asset case, prices at a single fixed horizon $T$ can also be sampled directly from the initial prices using one vector $\mathbf{Z}$ scaled by $\sqrt{T}$.

To use the model, we need to estimate $\mathbf{C}$ from data. Let's now construct the empirical growth-rate covariance matrix and use a scaling rule to obtain the covariance rate.

___

## Empirical Covariance Matrix
A covariance matrix records the variance of each feature and the pairwise covariance between features. Here, the features are the growth rates of the firms in $\mathcal{P}$, and each sample is an aligned vector of their growth rates over one period.

Let's consider samples from two-dimensional normal distributions with negative, zero, and positive covariance. The ellipses are contours of constant probability density, drawn at one and two standard deviations along the principal axes of each distribution, and their tilt shows the sign of the covariance. 


<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L5b-Covariance-Schematic.svg" width="880" alt="Three columns show samples with negative, zero, and positive covariance. In the top row each blue cloud carries a solid one standard deviation ellipse and a dashed two standard deviation ellipse whose tilt follows the sign of the covariance. The bottom row overlays each blue cloud with a red cloud whose covariance matrix is four times larger and draws the dashed two standard deviation ellipse of each cloud; the red ellipse is the blue cloud's ellipse scaled by two, so the standard deviations double while the correlation is preserved."/>
    </center>
</div>


In the lower row, each blue cloud is compared with a red cloud whose covariance matrix is four times larger. This doubles the standard deviations while preserving the correlation, so the dashed two standard deviation contour of the red cloud is the contour of the blue cloud scaled by two.

We build the estimate in three steps: define the pairwise sample covariance and correlation, compute all pairs at once from a data matrix, and convert the result to the covariance rate $\mathbf{C}$ that the model needs.

### Covariance and correlation
Suppose we have $N\geq2$ equally spaced (aligned) time periods of growth-rate data for the $M$ firms in $\mathcal{P}$.  Let $g_k^{(i)}$ be the growth rate of firm $i$ in period $k=1,2,\ldots,N$. We use $i$ and $j$ to index firms and $k$ to index time periods.

Collect firm $i$'s observations into the vector $\mathbf{g}^{(i)}=[g_1^{(i)},\ldots,g_N^{(i)}]^{\top}$ with the mean:
$$
g'_i=\frac{1}{N}\sum_{k=1}^{N}g_k^{(i)}.
$$
The empirical growth-rate covariance matrix $\hat{\mathbf{\Sigma}}_g\in\mathbb{R}^{M\times M}$ collects the pairwise sample covariances:
$$
\hat{\Sigma}_{g,ij}
=\frac{1}{N-1}\underbrace{\sum_{k=1}^{N}
\overbrace{\bigl(g_k^{(i)}-g'_i\bigr)}^{\text{deviation from mean}}
\bigl(g_k^{(j)}-g'_j\bigr)}_{\text{sum over time periods of product of deviations}},
\qquad i,j\in\mathcal{P}.
$$
When $i=j$, the two deviations are the same, so the diagonal entry $\hat{\Sigma}_{g,ii}$ is the sample variance of firm $i$'s growth rate. Its square root is the growth-rate standard deviation $\sigma_g$ from L3a (units: inverse years). An off-diagonal entry describes how two firms' growth rates vary together. 

We obtain the dimensionless __correlation__ between firms $i$ and $j$ by dividing the covariance by the product of the two standard deviations:
$$
\boxed{
\rho_{ij}
=\frac{\hat{\Sigma}_{g,ij}}{\sqrt{\hat{\Sigma}_{g,ii}\,\hat{\Sigma}_{g,jj}}}
\in[-1,1]
\quad\Longleftrightarrow\quad
\hat{\Sigma}_{g,ij}
=\rho_{ij}\sqrt{\hat{\Sigma}_{g,ii}\,\hat{\Sigma}_{g,jj}}.
}
$$
__Correlation__ compares the strength of linear relationships across pairs of firms. Zero correlation means no linear relationship in this sample; it does not imply that the firms' growth rates are independent. The covariance retains the growth-rate scales needed by the model.

### The data matrix and the sample covariance
To compute all pairwise covariances at once, arrange the growth rates in a __data matrix__ $\mathbf{G}\in\mathbb{R}^{N\times M}$. Rows are time periods and columns are firms, so row $k$ contains the growth rates of all $M$ firms over the same interval:
$$
\mathbf{G}=\begin{bmatrix}
g_1^{(1)} & g_1^{(2)} & \cdots & g_1^{(M)} \\
g_2^{(1)} & g_2^{(2)} & \cdots & g_2^{(M)} \\
\vdots & \vdots & \ddots & \vdots \\
g_N^{(1)} & g_N^{(2)} & \cdots & g_N^{(M)}
\end{bmatrix}.
$$
To center the data, subtract each firm's sample mean from its column. Let $\mathbf{g}^{\prime}=[g'_1,g'_2,\ldots,g'_M]^{\top}$ be the vector of sample means. The centered data matrix is given by:
$$
\tilde{\mathbf{G}}=\mathbf{G}-\mathbf{1}\,\mathbf{g}^{\prime\top},
$$
where $\mathbf{1}\in\mathbb{R}^{N}$ is a vector of ones. The product $\mathbf{1}\,\mathbf{g}^{\prime\top}$ is an $N\times M$ matrix with the sample means in every row. Column $i$ of $\tilde{\mathbf{G}}$ is the centered observation vector $\tilde{\mathbf{g}}^{(i)}=\mathbf{g}^{(i)}-g'_i\mathbf{1}$.

> __Outer product:__ The matrix $\mathbf{1}\,\mathbf{g}^{\prime\top}$ is an example of an outer product. The [outer product](https://en.wikipedia.org/wiki/Outer_product) of two vectors $\mathbf{a}\in\mathbb{R}^{N}$ and $\mathbf{b}\in\mathbb{R}^{M}$ is the $N\times M$ matrix $\mathbf{a}\mathbf{b}^{\top}$ with elements $(\mathbf{a}\mathbf{b}^{\top})_{kj}=a_kb_j$.

The empirical growth-rate covariance matrix can now be computed with one matrix product, whose entries are dot products of the centered columns:
$$
\boxed{
\hat{\mathbf{\Sigma}}_g
=\frac{1}{N-1}\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}
\quad\Longleftrightarrow\quad
\hat{\Sigma}_{g,ij}
=\frac{1}{N-1}\,\tilde{\mathbf{g}}^{(i)\top}\tilde{\mathbf{g}}^{(j)}
=\frac{1}{N-1}\sum_{k=1}^{N}\tilde{g}_k^{(i)}\tilde{g}_k^{(j)}.
}
$$
The $(i,j)$ entry of $\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}$ is the dot product of the centered columns for firms $i$ and $j$, and each term $\tilde{g}_k^{(i)}=g_k^{(i)}-g'_i$ is the deviation from the mean in period $k$. Dividing by $N-1$ therefore gives the same pairwise covariance defined above, for every pair of firms at once.

The centered-data formula also tells us what to expect from the estimate:

> __Covariance Matrix Properties:__
>
> * __Elements:__ The diagonal entries $\hat{\Sigma}_{g,ii}$ are the sample growth-rate variances, so they are non-negative. The off-diagonal entries $\hat{\Sigma}_{g,ij}$ are the sample covariances between firms $i$ and $j$.
> * __Symmetry:__ Swapping $i$ and $j$ leaves the products in the pairwise covariance formula unchanged, so $\hat{\Sigma}_{g,ij}=\hat{\Sigma}_{g,ji}$.
> * __Positive semidefinite:__ For any $\mathbf{v}\in\mathbb{R}^{M}$,
> $$
> \mathbf{v}^{\top}\hat{\mathbf{\Sigma}}_g\mathbf{v}
> =\frac{1}{N-1}\mathbf{v}^{\top}\tilde{\mathbf{G}}^{\top}\tilde{\mathbf{G}}\mathbf{v}
> =\frac{1}{N-1}\lVert\tilde{\mathbf{G}}\mathbf{v}\rVert_2^2
> \geq0,
> $$
> so the sample variance of any weighted sum of the growth rates is non-negative.

### From the growth-rate covariance to the GBM covariance rate
To estimate the covariance rate $\mathbf{C}$, we relate the growth rates of the model to the sample covariance we just computed. Dividing the one-step transition by $S_i(t_{k-1})$, taking logarithms, and dividing by $\Delta t$ gives the growth rate of asset $i$ over step $k$, and collecting the $M$ growth rates into a vector gives:
$$
\mathbf{g}=\boldsymbol{\mu}_g+\frac{1}{\sqrt{\Delta t}}\,\mathbf{A}\mathbf{Z}_k.
$$
Because $\mathbf{Z}_k$ has mean zero and identity covariance, the growth-rate vector of the model has covariance:
$$
\operatorname{Cov}(\mathbf{g})=\frac{1}{\Delta t}\,\mathbf{A}\mathbf{A}^{\top}=\frac{\mathbf{C}}{\Delta t}.
$$
Solving for $\mathbf{C}$ and substituting the sample covariance gives the estimate the price model needs:
$$
\boxed{
\hat{\mathbf{C}}=\Delta t\,\hat{\mathbf{\Sigma}}_g.
}
$$
The covariance rate has units of inverse years, and its diagonal holds the squared volatilities. Taking square roots gives:
$$
\hat{\sigma}_i=\sqrt{\hat{C}_{ii}}=\sqrt{\Delta t}\sqrt{\hat{\Sigma}_{g,ii}},
$$
which is L4b's relation $\hat{\sigma}=\sigma_g\sqrt{\Delta t}$ applied to each firm. For daily data, $\Delta t=1/252$ years, so $\hat{\mathbf{C}}$ is the growth-rate covariance divided by 252. Scaling by a positive constant cancels in the correlation formula, so $\hat{\mathbf{C}}$ and $\hat{\mathbf{\Sigma}}_g$ have the same correlation matrix.

Because $\hat{\mathbf{C}}=\Delta t\,\hat{\mathbf{\Sigma}}_g$ and $\Delta t$ is a positive scalar, the covariance rate $\hat{\mathbf{C}}$ is also symmetric and positive semidefinite. It is not always invertible: the centered data matrix has rank at most $N-1$, so the estimate is singular when $M\geq N$, which is why the model section allowed the symmetric square root in place of a Cholesky factor. The optional advanced material examines sampling error, shrinkage, and changing correlations.

Let's compute the empirical covariance matrix for the firms in our dataset and examine the relationships it describes.

> __Example__
>
> [▶ Compute the covariance matrix for our dataset](CHEME-5660-L5b-Example-CovarianceMatrix-Fall-2026.ipynb). We compute the empirical growth-rate covariance matrix and convert it to the GBM covariance rate. We check the covariance calculation, compare the corresponding volatilities with our L4b estimates, and interpret the covariance and correlation of a pair of firms.

The estimated covariance tells us how asset growth rates vary together. Next, let's describe how much of our investment to assign to each asset.

___


## Portfolio Weights and Dirichlet Sampling
Once we can simulate several assets together, the next question is how much of our money to put in each. Let total initial wealth be $W_0>0$ dollars, and let $\mathbf{w}=(w_1,\ldots,w_M)^{\top}$ contain the fractions invested in the assets at time zero. For a __fully invested, long-only__ portfolio, the weights satisfy:
$$
w_i\geq0,\qquad \sum_{i\in\mathcal{P}}w_i=1.
$$
These constraints define the $(M-1)$-dimensional __simplex__ $\Delta^{M-1}$: a line segment for two assets, a triangle for three, and so on.

The amount invested in asset $i$ is $w_iW_0$, so we buy $n_i=w_iW_0/S_i(0)$ shares. We allow fractional shares and ignore dividends and trading costs. Holding these share counts fixed, the __buy-and-hold__ portfolio wealth is given by:
$$
\boxed{
W_t=\sum_{i\in\mathcal{P}}n_iS_i(t)
=W_0\sum_{i\in\mathcal{P}}w_i\frac{S_i(t)}{S_i(0)}.
}
$$
The share counts remain fixed, but the fraction of wealth in each asset changes as relative prices move. At time $t$, that fraction is given by:
$$
w_i(t)=\frac{n_iS_i(t)}{W_t}.
$$
Thus, $\mathbf{w}$ specifies the initial allocation. Maintaining constant weights requires trading to rebalance the portfolio, with its own turnover and cost assumptions.

### Sampling candidate allocations
How should we choose $\mathbf{w}$? In L6a, we will solve an optimization problem. Today, we want to explore the simplex by drawing valid weight vectors and comparing the portfolios they produce. The Dirichlet distribution gives us a way to generate these candidates.

> __Dirichlet portfolio weights__
>
> Let $\boldsymbol{\alpha}=(\alpha_1,\ldots,\alpha_M)$ contain positive __concentration parameters__, and let $\alpha_0=\sum_{i\in\mathcal{P}}\alpha_i$. A random weight vector $\mathbf{W}\sim\operatorname{Dirichlet}(\boldsymbol{\alpha})$ has positive components with probability one, and they sum to one. We use a draw from this distribution as the initial allocation $\mathbf{w}$. The component moments are given by:
> $$
> \begin{aligned}
> \mathbb{E}[W_i]&=\frac{\alpha_i}{\alpha_0},\\
> \operatorname{Var}(W_i)&=\frac{\alpha_i(\alpha_0-\alpha_i)}{\alpha_0^2(\alpha_0+1)},\\
> \operatorname{Cov}(W_i,W_j)&=-\frac{\alpha_i\alpha_j}{\alpha_0^2(\alpha_0+1)},\qquad i\ne j.
> \end{aligned}
> $$
> The covariances are negative because the weights divide a fixed budget: a larger allocation to one asset leaves less for the others.

When every $\alpha_i=1$, the density is constant over the simplex, so regions of equal volume are equally likely. Each individual weight has a $\operatorname{Beta}(1,M-1)$ distribution, which is uniform only for two assets.

With symmetric concentrations $\alpha_i=\alpha$, values below one favor allocations near the boundary, where most of the budget is concentrated in a few assets. Large values cluster the draws near equal weights $1/M$. Unequal concentrations give larger expected allocations to assets with larger $\alpha_i$, as the mean formula shows. These choices let us explore different allocations before evaluating their growth and risk.

### Comparing growth and risk
For an allocation $\mathbf{w}$ held over one period of length $\Delta t$, the buy-and-hold wealth formula gives the exact portfolio growth rate:
$$
\frac{1}{\Delta t}\ln\!\left(\frac{W_{\Delta t}}{W_0}\right)
=\frac{1}{\Delta t}\ln\!\left(\sum_{i\in\mathcal{P}}w_i e^{r_i}\right),
$$
where $r_i=\ln(S_i(\Delta t)/S_i(0))=g_i\Delta t$ is asset $i$'s log return. To obtain a simpler measure for comparing allocations, retain only terms that are first order in these realized log returns. Using $e^{r_i}\approx1+r_i$, $\sum_iw_i=1$, and $\ln(1+x)\approx x$ gives:
$$
\frac{1}{\Delta t}\ln\!\left(\sum_{i\in\mathcal{P}}w_i e^{r_i}\right)
\approx\frac{1}{\Delta t}\sum_{i\in\mathcal{P}}w_i r_i
=\mathbf{w}^{\top}\mathbf{g}.
$$
We use $g_p=\mathbf{w}^{\top}\mathbf{g}$ as a __linear growth-rate proxy__. Recall that the multiple-asset model defined $\mu_{g,i}=\mathbb{E}[g_i]=\mu_i-C_{ii}/2$ as the mean growth rate of asset $i$. The vector $\boldsymbol{\mu}_g$ collects these model means for all assets. For fixed weights, the proxy's mean and variance are given by:
$$
\begin{aligned}
\mathbb{E}[g_p]&=\mathbf{w}^{\top}\mathbb{E}[\mathbf{g}]
=\mathbf{w}^{\top}\boldsymbol{\mu}_g,\\
\operatorname{Var}(g_p)&=\mathbf{w}^{\top}\operatorname{Cov}(\mathbf{g})\mathbf{w}.
\end{aligned}
$$
How do we evaluate these expressions from data? In the empirical covariance section, we calculated each firm's __sample mean__ from its $N$ historical growth-rate observations:
$$
g'_i=\frac{1}{N}\sum_{k=1}^{N}g_k^{(i)},
\qquad
\mathbf{g}^{\prime}=[g'_1,\ldots,g'_M]^{\top}.
$$
The prime denotes a sample mean. We use $\mathbf{g}^{\prime}$ to estimate the model mean vector $\boldsymbol{\mu}_g$, and $\hat{\mathbf{\Sigma}}_g$ to estimate $\operatorname{Cov}(\mathbf{g})$. Thus, for each candidate allocation $\mathbf{w}$, we compute:
$$
\begin{aligned}
\text{Estimated mean growth rate}&=\mathbf{w}^{\top}\mathbf{g}^{\prime},\\
\text{Estimated variance}&=\mathbf{w}^{\top}\hat{\mathbf{\Sigma}}_g\mathbf{w}.
\end{aligned}
$$
These estimates describe the linear proxy $g_p$. The exact portfolio log growth generally differs from the weighted sum of asset log growth, and buy-and-hold weights change over multiple periods. We use the wealth formula when following such a portfolio through time.

Sampling gives us candidate allocations to compare using these measures. The best sampled candidate need not be optimal over all feasible weights; finding an optimum is the topic of L6a. Let's examine how the concentration parameters change the candidates and the portfolios they produce.

> __Example__
>
> [▶ Sample portfolio weights with the Dirichlet distribution](CHEME-5660-L5b-Example-Dirichlet-PortfolioWeights-Fall-2026.ipynb). We draw long-only allocations, examine how the concentration parameters shape the weights, and compare the estimated growth and risk of the sampled portfolios. We also follow selected buy-and-hold portfolios to see how their wealth and weights change as prices move.

This comparison prepares us to choose weights using an explicit growth and risk objective in L6a.

___


## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L6a; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ Derivation of the multiple asset GBM solution](advanced/ito-derivation/CHEME-5660-L5b-Derivation-MAGBM-Solution-Fall-2026.ipynb). Why does each asset's drift correction contain only its own variance rate $C_{ii}$ and none of the covariances? We state the multiplication rules for the increments of independent Wiener processes, extend Itô's lemma from L4b to several noise sources, and apply it to the log price of each asset. Integrating on the time grid recovers the boxed one-step transition, and writing the growth rates in vector form gives $\operatorname{Cov}(\mathbf{g})=\mathbf{C}/\Delta t$, the scaling relation used to estimate the covariance rate from data.

* [▶ Sampling error and shrinkage in covariance estimation](advanced/covariance-estimation/CHEME-5660-L5b-Advanced-CovarianceEstimation-Fall-2026.ipynb). How does uncertainty in a covariance estimate affect portfolio selection? We compare the sample correlation eigenvalues with a reference for independent data, the Marchenko–Pastur law, and measure estimation error using simulated samples. We then introduce shrinkage, which combines the sample covariance with a simpler target matrix, and compare the resulting minimum-variance portfolios on a later data window.

* [▶ Rolling correlations](advanced/rolling-correlation/CHEME-5660-L5b-Advanced-RollingCorrelation-Fall-2026.ipynb). Are correlations between firms stable over time? We estimate correlations from 2014 to 2024 using rolling windows and exponentially weighted observations, then compare them with the full-sample estimates. We examine how average correlation changes during periods of high market volatility and what these changes mean for a model with a constant covariance rate.

___



## Summary
In this lecture, we extended geometric Brownian motion from one asset to many correlated assets, used historical growth rates to estimate their covariance, and introduced Dirichlet sampling as a way to explore portfolio weights on the long-only simplex.

> __Key Takeaways:__
>
> * **Correlated asset prices:** We extended single asset geometric Brownian motion by using a loading matrix to combine independent Wiener increments. This gave us a model with a specified covariance rate, a mean growth rate for each asset, and an exact transition for generating correlated price paths.
>
> * **Covariance from data:** We constructed the empirical covariance matrix from centered growth rates and multiplied it by the time step to obtain the model's covariance rate. This connects historical growth-rate variation to asset volatilities and co-movement. We also examined why a valid covariance estimate can be singular or sensitive to the data window.
>
> * **Portfolio weights:** We used the Dirichlet distribution to explore long-only allocations on the simplex and compared candidates using the estimated mean and variance of a linear growth-rate proxy. We also expressed buy-and-hold wealth in terms of fixed share counts, which explains why investment weights change as prices move.

Next time, we turn the estimated means and covariance into the minimum-variance portfolio and the efficient frontier.

___



## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
